# Notebook 10 — Protein Quantification Methods

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 10 of 12  
**Time estimate:** 75 minutes

> Identifying proteins is solved. Quantifying them accurately across samples is hard.
> Label-free, TMT, and SILAC represent three generations of the same problem:
> how do you compare protein abundance between conditions?

---
## Step 1 — Motivation

After protein identification (NB09), the next question is: which proteins change
in abundance between conditions? This requires accurate quantification that is
comparable across samples — a challenge because LC-MS/MS has variable sensitivity
run-to-run.

---
## Step 2 — Intuition

**Three approaches:**

1. **Spectral counting (label-free):** Count the number of PSMs per protein per sample.
   More abundant proteins generate more spectra. Simple but imprecise.

2. **LFQ intensity (label-free):** Use the MS1 precursor ion intensity (area under
   the chromatographic peak). MaxLFQ algorithm (Cox 2014) normalizes across samples
   by finding proteins that are constant across samples and using them as a reference.

3. **TMT / iTRAQ (isobaric labeling):** Chemically label all peptides in each sample
   with a different mass tag. Pool all samples, run one LC-MS/MS experiment. At MS2,
   the tags fragment to generate reporter ions at different m/z — ratios between
   reporter intensities give relative quantities. Higher precision, higher cost.

4. **SILAC (metabolic labeling):** Cells grown in heavy (13C/15N) amino acids.
   Proteins from treated/control cells differ in mass by a fixed amount — compared
   in the same MS1 scan. Most accurate but requires cell culture.

---
## Step 3 — Biological Background

**MaxLFQ normalization (Cox 2014):**  
Key insight: most proteins do not change between conditions. Proteins consistently
quantified across samples can be used to compute a normalization factor that aligns
samples. MaxLFQ implements a least-squares approach to estimate protein intensities
from pairwise sample comparisons, maximizing the number of proteins used.

**TMT reporter ions:**  
TMT tags have equal total mass but different distributions between a reporter group
(which fragments to distinct m/z) and a balancer group. All tags give the same
precursor mass (isobaric) but different reporter ion masses (128.1, 129.1, 130.1...)

**Missing value imputation strategies:**
- **MinProb:** Sample from a truncated normal distribution at the low end of the
  intensity distribution — models proteins near the detection limit
- **KNN:** Impute from k most similar proteins by expression profile
- **Mean/median:** simple but ignores the non-random structure of missing data
- **No imputation:** drop proteins with any missing value (conservative but loses data)

---
## Step 4 — Mathematical Explanation

**MaxLFQ core idea (simplified):**  
For protein $p$, pair of samples $(i, j)$, observed log-ratio:
$$r_{ij}^p = \log_2(I_{pi}) - \log_2(I_{pj})$$

MaxLFQ finds normalized log-intensities $\hat{q}_{pi}$ minimizing:
$$\sum_{p} \sum_{i < j} w_{pij} (\hat{q}_{pi} - \hat{q}_{pj} - r_{ij}^p)^2$$

where $w_{pij}$ = number of peptides supporting this ratio.

**Downstream DE testing (t-test on log2 LFQ intensities):**
$$t = \frac{\bar{x}_{\text{treated}} - \bar{x}_{\text{control}}}{s_p \sqrt{1/n_1 + 1/n_2}}$$

where $s_p$ is the pooled standard deviation. After BH correction, proteins with
adj. p < 0.05 and |log2FC| > 1 are reported as differentially abundant.

In [ ]:
# Step 6 — Python: LFQ normalization, imputation, and differential abundance
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ---- Simulate MaxQuant-like proteinGroups output ----
N_PROTEINS = 500
N_SAMPLES = 6  # 3 control, 3 treatment samples: C1, C2, C3, T1, T2, T3
CONDITIONS = ['control'] * 3 + ['treatment'] * 3

# Baseline (log2 LFQ intensity)
baseline = rng.normal(24, 3, N_PROTEINS)
# 50 differentially abundant proteins
DE_IDX = rng.choice(N_PROTEINS, 50, replace=False)
DE_EFFECT = rng.choice([-2.5, -2.0, 2.0, 2.5, 3.0], 50)

# Generate intensities
LFQ = np.zeros((N_PROTEINS, N_SAMPLES))
for j, cond in enumerate(CONDITIONS):
    noise = rng.normal(0, 0.4, N_PROTEINS)
    if cond == 'control':
        LFQ[:, j] = baseline + noise
    else:
        LFQ[:, j] = baseline + noise
        LFQ[DE_IDX, j] += DE_EFFECT

# Add systematic sample loading variation (a.k.a. batch effect on scale)
scale_factors = rng.uniform(0.8, 1.3, N_SAMPLES)  # ±30% loading variation
LFQ_biased = LFQ + np.log2(scale_factors)[np.newaxis, :]

# Add missing values (~20%, biased towards low-intensity proteins)
LFQ_missing = LFQ_biased.copy()
for i in range(N_PROTEINS):
    for j in range(N_SAMPLES):
        # Low-intensity proteins missing more often
        miss_prob = 0.05 + 0.4 * (1 - (baseline[i] - baseline.min()) / (baseline.max() - baseline.min()))
        if rng.random() < miss_prob:
            LFQ_missing[i, j] = np.nan

pct_missing = np.isnan(LFQ_missing).mean() * 100
print(f'Missing values: {pct_missing:.1f}%')

# ---- Median normalization (simple proxy for MaxLFQ) ----
def median_normalize(X):
    """Subtract sample median from each column (in log space)."""
    # Only use proteins with no missing values for normalization
    complete_rows = ~np.any(np.isnan(X), axis=1)
    sample_medians = np.nanmedian(X[complete_rows, :], axis=0)
    global_median  = np.nanmedian(sample_medians)
    shift = global_median - sample_medians
    return X + shift[np.newaxis, :]

LFQ_norm = median_normalize(LFQ_missing)

# ---- MinProb imputation ----
def minprob_impute(X, width=0.3, shift=1.8):
    """Impute missing values with values from low end of distribution."""
    X_imp = X.copy()
    for j in range(X.shape[1]):
        col = X[:, j]
        observed = col[~np.isnan(col)]
        if len(observed) < 10:
            continue
        q01 = np.percentile(observed, 2.5)
        sigma = observed.std() * width
        # Impute from N(q01, sigma)
        n_missing = np.isnan(col).sum()
        X_imp[np.isnan(col), j] = rng.normal(q01 - shift * sigma, sigma, n_missing)
    return X_imp

LFQ_imputed = minprob_impute(LFQ_norm)

# ---- Differential abundance: t-test with BH correction ----
def de_analysis_proteomics(LFQ, conditions):
    """Welch t-test per protein, BH correction."""
    ctrl_idx = [i for i, c in enumerate(conditions) if c == 'control']
    trt_idx  = [i for i, c in enumerate(conditions) if c == 'treatment']
    n_prot = LFQ.shape[0]
    pvals  = np.ones(n_prot)
    log2fc = np.zeros(n_prot)
    for i in range(n_prot):
        a = LFQ[i, ctrl_idx]
        b = LFQ[i, trt_idx]
        if np.std(a) < 1e-9 and np.std(b) < 1e-9:
            continue
        t, p = stats.ttest_ind(b, a)
        pvals[i]  = p
        log2fc[i] = b.mean() - a.mean()
    # BH correction
    m = n_prot
    rank = np.argsort(pvals)
    adj = np.ones(m)
    for i, r in enumerate(rank):
        adj[r] = min(pvals[r] * m / (i + 1), 1.0)
    min_sf = 1.0
    for r in rank[::-1]:
        adj[r] = min(adj[r], min_sf)
        min_sf = adj[r]
    return log2fc, adj

log2fc, adj_pval = de_analysis_proteomics(LFQ_imputed, CONDITIONS)
sig_mask = (adj_pval < 0.05) & (np.abs(log2fc) > 1.0)
tp = np.intersect1d(np.where(sig_mask)[0], DE_IDX)
fp = len(np.where(sig_mask)[0]) - len(tp)
print(f'\nDE results: {sig_mask.sum()} significant proteins')
print(f'True positives: {len(tp)}/50 ({len(tp)/50*100:.0f}%)')
print(f'False positives: {fp}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Before vs. after normalization (boxplot per sample)
ax = axes[0]
data_before = [LFQ_biased[:, j][~np.isnan(LFQ_biased[:, j])] for j in range(N_SAMPLES)]
data_after  = [LFQ_norm[:, j][~np.isnan(LFQ_norm[:, j])] for j in range(N_SAMPLES)]
bp1 = ax.boxplot(data_before, positions=range(1, 7), widths=0.3, patch_artist=True,
                   boxprops={'facecolor': 'steelblue', 'alpha': 0.5})
bp2 = ax.boxplot(data_after, positions=np.arange(1, 7) + 0.4, widths=0.3, patch_artist=True,
                   boxprops={'facecolor': 'tomato', 'alpha': 0.5})
ax.set_xlabel('Sample')
ax.set_ylabel('log2 LFQ intensity')
ax.set_xticks(np.arange(1, 7) + 0.2)
ax.set_xticklabels(['C1','C2','C3','T1','T2','T3'], fontsize=8)
ax.set_title('A. Before (blue) vs. after (red)\nmedian normalization')

# Panel B: Intensity distribution + MinProb imputation
ax = axes[1]
observed_vals = LFQ_norm[~np.isnan(LFQ_norm)]
imputed_vals  = LFQ_imputed[np.isnan(LFQ_norm)]
ax.hist(observed_vals, bins=60, density=True, alpha=0.6, label='Observed', color='steelblue')
ax.hist(imputed_vals,  bins=40, density=True, alpha=0.6, label='Imputed (MinProb)', color='tomato')
ax.set_xlabel('log2 LFQ intensity')
ax.set_ylabel('Density')
ax.set_title('B. MinProb imputation\n(imputed values at low end)')
ax.legend(fontsize=8)

# Panel C: Volcano plot
ax = axes[2]
nlp = -np.log10(adj_pval + 1e-300)
colors_vol = []
for i in range(N_PROTEINS):
    if adj_pval[i] < 0.05 and log2fc[i] > 1.0:
        colors_vol.append('tomato')  # up
    elif adj_pval[i] < 0.05 and log2fc[i] < -1.0:
        colors_vol.append('steelblue')  # down
    else:
        colors_vol.append('lightgrey')  # not significant
ax.scatter(log2fc, nlp, c=colors_vol, s=8, alpha=0.7)
ax.axhline(-np.log10(0.05), color='black', ls='--', lw=0.8)
ax.axvline(1.0, color='grey', ls='--', lw=0.8)
ax.axvline(-1.0, color='grey', ls='--', lw=0.8)
ax.set_xlabel('log2 Fold Change (treatment/control)')
ax.set_ylabel('-log10(adjusted p-value)')
ax.set_title(f'C. Volcano plot\n({sig_mask.sum()} significant, {len(tp)}/50 true positives)')

plt.suptitle('Module 10 NB10: Protein Quantification Methods', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('proteomics_quantification.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `median_normalize()` and verify that all sample medians are equal after
   normalization.
2. What happens to the volcano plot if you skip normalization?
3. Compare MinProb imputation to mean imputation (replace NaN with column mean).
   Which recovers more true DE proteins?
4. In TMT experiments, all samples are mixed before MS analysis. What systematic
   error is eliminated by this design compared to label-free?

---
## Step 10 — Quiz

1. What is spectral counting and what is its main limitation?
2. What is the key assumption behind MaxLFQ normalization?
3. Why is MinProb a better imputation strategy than mean imputation for proteomics?
4. What does the TMT reporter ion intensity represent?
5. A protein has log2FC = 2.5 but is missing in 2 of 3 control replicates. Is this
   a reliable differential abundance result?

---
## Step 12 — Reflection

> *[In one paragraph: compare LFQ and TMT quantification in terms of precision,
> sample throughput, and appropriate experimental design, and explain when
> you would choose each.]*

---
*Next: `11_multiomics_integration_survey.ipynb`*